# 01 — 合成数据与组合环境

这个 notebook 展示当前 demo 中哪些数据是合成的，以及一次环境 `step` 如何把动作变成目标组合、扣除费用并结算下一期收益。

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

repo_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src/res/rl_experimental').exists())
sys.path.insert(0, str(repo_root))

from src.res.rl_experimental.data import make_synthetic_panel  # noqa: E402
from src.res.rl_experimental.env import EpisodeRange, PortfolioEnv  # noqa: E402
from src.res.rl_experimental.portfolio import PortfolioConstraints  # noqa: E402

panel = make_synthetic_panel(n_steps=160, n_stocks=60, n_features=8, seed=7)
print('features:', panel.features.shape)
print('alpha:', panel.alpha.shape)
print('returns:', panel.forward_returns.shape)
print('investable ratio:', panel.investable.mean().round(3))

ModuleNotFoundError: No module named 'gymnasium'

合成器生成市场共同收益、行业共同收益、个股噪声和弱 alpha 信号。下面检查 alpha 与下一期收益之间只有较弱的正相关，而不是一个容易被模型机械破解的数据集。

In [ ]:
mask = panel.investable & np.isfinite(panel.forward_returns)
correlation = np.corrcoef(panel.alpha[mask], panel.forward_returns[mask])[0, 1]
print('pooled alpha/forward-return correlation:', round(float(correlation), 4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(panel.alpha[mask][::20], panel.forward_returns[mask][::20], s=5, alpha=.25)
axes[0].set(xlabel='synthetic alpha', ylabel='next-period return', title='Weak predictive signal')
axes[1].plot(panel.investable.sum(axis=1))
axes[1].set(xlabel='day', ylabel='eligible stocks', title='Changing investable universe')
plt.tight_layout()

环境使用 20 个 alpha 候选和最多 10 个旧持仓槽位。`present_mask` 屏蔽空槽，`investable_mask` 决定哪些股票可以获得新的目标权重。

In [ ]:
constraints = PortfolioConstraints(candidate_count=20, max_holdings=10, max_weight=0.12)
env = PortfolioEnv(panel, EpisodeRange(0, 80), constraints)
obs, info = env.reset(seed=1)
print('fixed slots:', constraints.max_slots)
print('real stocks in slots:', int(obs['present_mask'].sum()))
print('currently investable:', int(obs['investable_mask'].sum()))
print('first slot stock IDs:', info['stock_ids'][:8])

In [ ]:
# 零动作表示：不修正 alpha 排名，配权偏好相同，因此得到 alpha 前 10 等权组合。
action = np.zeros(env.action_space.shape, dtype=np.float32)
obs, reward, terminated, truncated, info = env.step(action)
print('selected IDs:', info['selected_stock_ids'])
print('target weights:', np.round(info['target_weights'], 4))
print('turnover:', round(info['turnover'], 4), 'fee:', round(info['fee_estimate'], 6))
print('period return:', round(info['period_return'], 5), 'reward:', round(reward, 5))
print('post-return held weights:', np.round(info['held_weights'], 4))

In [ ]:
# 连续运行 alpha 等权基线，观察净值、换手和持仓数。
while not terminated:
    obs, reward, terminated, truncated, info = env.step(action)
nav = np.array([row['nav'] for row in env.history])
turnover = np.array([row['turnover'] for row in env.history])
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(nav)
axes[0].set_ylabel('NAV')
axes[1].plot(turnover)
axes[1].set_ylabel('turnover')
axes[1].set_xlabel('step')
plt.tight_layout()